In [1]:
import torch
from torch import nn
import torch.nn.functional as F
from dataclasses import dataclass

In [2]:
@dataclass
class GLM4MoeConfig:
    vocab_size: int = 151552
    hidden_size: int = 5120
    num_hidden_layers: int = 92
    rms_norm_eps: float = 1e-5
    num_attention_heads: int = 96
    num_key_value_heads: int = 8
    head_dim: int = 128
    attention_bias: bool = True
    use_qk_norm: bool = True
    rope_theta: float = 1_000_000
    partial_rotary_factor: float = 0.5
    intermediate_size: int = 12288
    hidden_act: type[nn.Module] = nn.SiLU
    first_k_dense_replace: int = 3
    n_routed_experts: int = 160
    num_experts_per_tok: int = 8
    moe_intermediate_size: int = 1536
    n_shared_experts: int = 1
    routed_scaling_factor: float = 2.5
    norm_topk_prob: bool = True
    n_group: int = 1
    topk_group: int = 1

In [3]:
tiny = GLM4MoeConfig(
    vocab_size=256,
    hidden_size=64,
    num_hidden_layers=6,  # Dense 3 + MoE 3
    num_attention_heads=4,
    num_key_value_heads=2,  # GQA 2:1
    head_dim=32,
    intermediate_size=128,
    moe_intermediate_size=48,
    n_routed_experts=8,
    num_experts_per_tok=2,
    routed_scaling_factor=2.5,
    rope_theta=10000,
)

In [4]:
class KVCache:
    def __init__(self):
        self.cache: list[tuple[torch.Tensor, torch.Tensor]] = []

    def update(self, layer_idx: int, k: torch.Tensor, v: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # k, v: (B, H_kv, L, D)
        if layer_idx < len(self.cache):
            prev_k, prev_v = self.cache[layer_idx]
            # (B, H_kv, L_prev + L, D)
            k = torch.cat([prev_k, k], dim=2)
            v = torch.cat([prev_v, v], dim=2)
            self.cache[layer_idx] = (k, v)
        else:
            self.cache.append((k, v))
        return k, v

    @property
    def seq_length(self) -> int:
        # (B, H_kv, L, D)
        return self.cache[0][0].shape[2] if self.cache else 0

In [5]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor):
        input_dtype = x.dtype
        x = x.to(torch.float32)
        x = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return self.weight * x.to(input_dtype)

In [6]:
class RotaryEmbedding(nn.Module):
    def __init__(self, config: GLM4MoeConfig):
        super().__init__()
        dim = int(config.head_dim * config.partial_rotary_factor)
        inv_freq = 1.0 / (config.rope_theta ** (torch.arange(0, dim, 2).to(torch.float32) / dim))
        self.register_buffer("inv_freq", inv_freq)

    def forward(self, x: torch.Tensor, position_ids: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # (B, D/4, 1)
        inv_freq = self.inv_freq[None, :, None].expand(position_ids.shape[0], -1, 1)
        # (B, 1, L)
        pos = position_ids[:, None, :].to(torch.float32)
        # (B, L, D/4)
        freqs = (inv_freq.to(torch.float32) @ pos).transpose(1, 2)
        # (B, L, D/2)
        emb = torch.cat((freqs, freqs), dim=-1)
        return emb.cos().to(x.dtype), emb.sin().to(x.dtype)

In [7]:
def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

In [8]:
def apply_rotary_pos_emb(
    q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor
) -> tuple[torch.Tensor, torch.Tensor]:
    # q: (B, H, L, D), k: (B, H_kv, L, D)
    # (B, 1, L, D/2)
    cos = cos.unsqueeze(1)
    sin = sin.unsqueeze(1)
    rotary_dim = cos.shape[-1]

    # (B, H, L, D/2), (B, H, L, D/2)
    q_rot, q_pass = q[..., :rotary_dim], q[..., rotary_dim:]
    # (B, H_kv, L, D/2), (B, H_kv, L, D/2)
    k_rot, k_pass = k[..., :rotary_dim], k[..., rotary_dim:]

    q_embed = (q_rot * cos) + (rotate_half(q_rot) * sin)
    k_embed = (k_rot * cos) + (rotate_half(k_rot) * sin)

    # (B, H, L, D), (B, H_kv, L, D)
    q = torch.cat([q_embed, q_pass], dim=-1)
    k = torch.cat([k_embed, k_pass], dim=-1)

    return q, k

In [9]:
class Attention(nn.Module):
    def __init__(self, config: GLM4MoeConfig, layer_idx: int):
        super().__init__()
        self.layer_idx = layer_idx
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.num_kv_groups = self.num_heads // self.num_kv_heads
        self.head_dim = config.head_dim
        self.scaling = self.head_dim**-0.5

        self.q_proj = nn.Linear(config.hidden_size, self.num_heads * self.head_dim, bias=config.attention_bias)
        self.k_proj = nn.Linear(config.hidden_size, self.num_kv_heads * self.head_dim, bias=config.attention_bias)
        self.v_proj = nn.Linear(config.hidden_size, self.num_kv_heads * self.head_dim, bias=config.attention_bias)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, config.hidden_size, bias=False)

        self.use_qk_norm = config.use_qk_norm
        if self.use_qk_norm:
            self.q_norm = RMSNorm(self.head_dim, config.rms_norm_eps)
            self.k_norm = RMSNorm(self.head_dim, config.rms_norm_eps)

    def forward(
        self,
        x: torch.Tensor,
        cos: torch.Tensor,
        sin: torch.Tensor,
        mask: torch.Tensor | None = None,
        cache: KVCache | None = None,
    ) -> torch.Tensor:
        B, L, _ = x.shape

        q = self.q_proj(x).view(B, L, self.num_heads, self.head_dim)
        k = self.k_proj(x).view(B, L, self.num_kv_heads, self.head_dim)
        v = self.v_proj(x).view(B, L, self.num_kv_heads, self.head_dim)

        if self.use_qk_norm:
            q = self.q_norm(q)
            k = self.k_norm(k)

        # (B, H, L, D)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        q, k = apply_rotary_pos_emb(q, k, cos, sin)

        if cache is not None:
            k, v = cache.update(self.layer_idx, k, v)

        # GQA: (B, H_kv, L, D) → (B, H, L, D) via expand (no copy)
        B, H_kv, L_kv, D = k.shape
        k = k.unsqueeze(2).expand(B, H_kv, self.num_kv_groups, L_kv, D).reshape(B, -1, L_kv, D)
        v = v.unsqueeze(2).expand(B, H_kv, self.num_kv_groups, L_kv, D).reshape(B, -1, L_kv, D)

        attn = (q @ k.transpose(-2, -1)) * self.scaling
        if mask is not None:
            attn = attn + mask
        attn = attn.softmax(dim=-1, dtype=torch.float32).to(q.dtype)

        out = (attn @ v).transpose(1, 2).reshape(B, L, -1)
        out = self.o_proj(out)
        return out

In [23]:
class MLP(nn.Module):
    def __init__(self, config: GLM4MoeConfig, intermediate_size: int | None = None):
        super().__init__()
        intermediate_size = intermediate_size or config.intermediate_size
        self.gate_proj = nn.Linear(config.hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, config.hidden_size, bias=False)
        self.act = config.hidden_act()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate = self.gate_proj(x)
        up = self.up_proj(x)
        out = self.down_proj(self.act(gate) * up)
        return out

In [10]:
position_ids = torch.arange(8)[None, :].expand(2, -1)

In [11]:
position_ids.shape

torch.Size([2, 8])

In [12]:
pos = position_ids[:, None, :].to(torch.float32)
pos.shape

torch.Size([2, 1, 8])

In [13]:
inv_freq = 1.0 / (1000 ** (torch.arange(0, 8, 2).to(torch.float32) / 8))

In [14]:
inv_freq

tensor([1.0000, 0.1778, 0.0316, 0.0056])

In [15]:
inv_freq.shape

torch.Size([4])

In [16]:
_inv_freq = inv_freq[None, :, None].expand(position_ids.shape[0], -1, 1)

In [17]:
_inv_freq.shape

torch.Size([2, 4, 1])

In [18]:
(_inv_freq @ pos).shape

torch.Size([2, 4, 8])

In [19]:
freqs = (_inv_freq @ pos).transpose(1, 2)

In [20]:
freqs

tensor([[[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [1.0000e+00, 1.7783e-01, 3.1623e-02, 5.6234e-03],
         [2.0000e+00, 3.5566e-01, 6.3246e-02, 1.1247e-02],
         [3.0000e+00, 5.3348e-01, 9.4868e-02, 1.6870e-02],
         [4.0000e+00, 7.1131e-01, 1.2649e-01, 2.2494e-02],
         [5.0000e+00, 8.8914e-01, 1.5811e-01, 2.8117e-02],
         [6.0000e+00, 1.0670e+00, 1.8974e-01, 3.3740e-02],
         [7.0000e+00, 1.2448e+00, 2.2136e-01, 3.9364e-02]],

        [[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [1.0000e+00, 1.7783e-01, 3.1623e-02, 5.6234e-03],
         [2.0000e+00, 3.5566e-01, 6.3246e-02, 1.1247e-02],
         [3.0000e+00, 5.3348e-01, 9.4868e-02, 1.6870e-02],
         [4.0000e+00, 7.1131e-01, 1.2649e-01, 2.2494e-02],
         [5.0000e+00, 8.8914e-01, 1.5811e-01, 2.8117e-02],
         [6.0000e+00, 1.0670e+00, 1.8974e-01, 3.3740e-02],
         [7.0000e+00, 1.2448e+00, 2.2136e-01, 3.9364e-02]]])

In [21]:
emb = torch.cat((freqs, freqs), dim=-1)
emb

tensor([[[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
          0.0000e+00, 0.0000e+00, 0.0000e+00],
         [1.0000e+00, 1.7783e-01, 3.1623e-02, 5.6234e-03, 1.0000e+00,
          1.7783e-01, 3.1623e-02, 5.6234e-03],
         [2.0000e+00, 3.5566e-01, 6.3246e-02, 1.1247e-02, 2.0000e+00,
          3.5566e-01, 6.3246e-02, 1.1247e-02],
         [3.0000e+00, 5.3348e-01, 9.4868e-02, 1.6870e-02, 3.0000e+00,
          5.3348e-01, 9.4868e-02, 1.6870e-02],
         [4.0000e+00, 7.1131e-01, 1.2649e-01, 2.2494e-02, 4.0000e+00,
          7.1131e-01, 1.2649e-01, 2.2494e-02],
         [5.0000e+00, 8.8914e-01, 1.5811e-01, 2.8117e-02, 5.0000e+00,
          8.8914e-01, 1.5811e-01, 2.8117e-02],
         [6.0000e+00, 1.0670e+00, 1.8974e-01, 3.3740e-02, 6.0000e+00,
          1.0670e+00, 1.8974e-01, 3.3740e-02],
         [7.0000e+00, 1.2448e+00, 2.2136e-01, 3.9364e-02, 7.0000e+00,
          1.2448e+00, 2.2136e-01, 3.9364e-02]],

        [[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.00

In [22]:
emb.cos()

tensor([[[ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,
           1.0000],
         [ 0.5403,  0.9842,  0.9995,  1.0000,  0.5403,  0.9842,  0.9995,
           1.0000],
         [-0.4161,  0.9374,  0.9980,  0.9999, -0.4161,  0.9374,  0.9980,
           0.9999],
         [-0.9900,  0.8610,  0.9955,  0.9999, -0.9900,  0.8610,  0.9955,
           0.9999],
         [-0.6536,  0.7575,  0.9920,  0.9997, -0.6536,  0.7575,  0.9920,
           0.9997],
         [ 0.2837,  0.6301,  0.9875,  0.9996,  0.2837,  0.6301,  0.9875,
           0.9996],
         [ 0.9602,  0.4828,  0.9821,  0.9994,  0.9602,  0.4828,  0.9821,
           0.9994],
         [ 0.7539,  0.3203,  0.9756,  0.9992,  0.7539,  0.3203,  0.9756,
           0.9992]],

        [[ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,
           1.0000],
         [ 0.5403,  0.9842,  0.9995,  1.0000,  0.5403,  0.9842,  0.9995,
           1.0000],
         [-0.4161,  0.9374,  0.9980,  0.9999, -0.4161,  0.9374,  0.9